In [8]:
import os
import shutil
import numpy as np
import pandas as pd
from scipy.integrate import solve_ivp

q = 0.5

num_train = 160
num_var = 20
num_test = 20
num_total = num_train + num_var + num_test

seq_len = 500
high_dim = 64

t_end = 20.0

T_burn = 0.5
burn_len = 100

F0_range = (0.3, 1.2)
omega_range = (0.7, 2.2)
phi_range = (0.0, 2.0 * np.pi)

x0_abs_range = (0.9, 2.4)
v0_range = (-1.2, 1.2)

max_abs_x_allowed = 2.85

min_x_range_allowed = 1.1
min_rms_x_allowed = 0.75

large_angle_threshold = 0.8
min_large_angle_fraction = 0.30

min_nonlinear_gap_allowed = 0.12

max_trials = 30000

max_harmonic = 3

root_dir = "pendulum_dataset_periodic_lift_large_angle"
train_dir = os.path.join(root_dir, "train")
var_dir = os.path.join(root_dir, "var")
test_dir = os.path.join(root_dir, "test")

clear_existing = True

seed = 42
rng = np.random.default_rng(seed)


if clear_existing and os.path.exists(root_dir):
    shutil.rmtree(root_dir)

os.makedirs(train_dir, exist_ok=True)
os.makedirs(var_dir, exist_ok=True)
os.makedirs(test_dir, exist_ok=True)


if high_dim < 2:
    raise ValueError("high_dim should be at least 2.")

num_pairs = high_dim // 2

k_base = np.arange(1, max_harmonic + 1, dtype=np.int64)
harmonics = np.resize(k_base, num_pairs)
rng.shuffle(harmonics)

phases = rng.uniform(0.0, 2.0 * np.pi, size=(num_pairs,))

extra_harmonic = int(rng.integers(1, max_harmonic + 1))
extra_phase = float(rng.uniform(0.0, 2.0 * np.pi))


def lift_to_high_dim(x):
    x = np.asarray(x, dtype=np.float64)
    channels = []

    for k, phase in zip(harmonics, phases):
        arg = float(k) * x + float(phase)
        channels.append(np.sin(arg))
        channels.append(np.cos(arg))

    if len(channels) < high_dim:
        arg = float(extra_harmonic) * x + float(extra_phase)
        channels.append(np.sin(arg))

    X_high = np.asarray(channels[:high_dim], dtype=np.float32)
    return X_high


# ======================
# x'' + q x' + sin(x) = F0 cos(omega t + phi)
# ======================
def forced_pendulum(t, y, F0, omega, phi):
    x = y[0]
    v = y[1]

    dxdt = v
    dvdt = F0 * np.cos(omega * t + phi) - q * v - np.sin(x)

    return [dxdt, dvdt]


def sample_burn_initial_state():
    sign = rng.choice([-1.0, 1.0])
    x0 = sign * rng.uniform(*x0_abs_range)
    v0 = rng.uniform(*v0_range)

    return [float(x0), float(v0)]


def simulate_one_candidate(global_id):
    F0 = rng.uniform(*F0_range)
    omega = rng.uniform(*omega_range)
    phi = rng.uniform(*phi_range)

    t_burn = np.linspace(-T_burn, 0.0, burn_len, dtype=np.float64)

    y_burn0 = sample_burn_initial_state()

    sol_burn = solve_ivp(
        fun=lambda tt, yy: forced_pendulum(tt, yy, F0, omega, phi),
        t_span=(t_burn[0], t_burn[-1]),
        y0=y_burn0,
        t_eval=t_burn,
        method="RK45",
        rtol=1e-9,
        atol=1e-9,
    )

    if not sol_burn.success:
        raise RuntimeError(f"Burn-in integration failed at global_id={global_id}")

    y0_formal = sol_burn.y[:, -1].copy()

    t = np.linspace(0.0, t_end, seq_len, dtype=np.float64)
    f = F0 * np.cos(omega * t + phi)

    sol = solve_ivp(
        fun=lambda tt, yy: forced_pendulum(tt, yy, F0, omega, phi),
        t_span=(t[0], t[-1]),
        y0=y0_formal,
        t_eval=t,
        method="RK45",
        rtol=1e-9,
        atol=1e-9,
    )

    if not sol.success:
        raise RuntimeError(f"Formal integration failed at global_id={global_id}")

    x = sol.y[0].astype(np.float64)
    v = sol.y[1].astype(np.float64)

    x_abs_max = float(np.max(np.abs(x)))
    x_range = float(np.max(x) - np.min(x))
    v_abs_max = float(np.max(np.abs(v)))
    rms_x = float(np.sqrt(np.mean(x ** 2)))

    large_angle_fraction = float(np.mean(np.abs(x) > large_angle_threshold))

    nonlinear_gap = np.sin(x) - x
    nonlinear_gap_rel = float(
        np.sqrt(np.mean(nonlinear_gap ** 2)) /
        (np.sqrt(np.mean(np.sin(x) ** 2)) + 1e-12)
    )

    if np.std(x) < 1e-12 or np.std(np.sin(x)) < 1e-12:
        corr_x_sinx = np.nan
    else:
        corr_x_sinx = float(np.corrcoef(x, np.sin(x))[0, 1])

    X_high = lift_to_high_dim(x)

    sample = {
        "t": t.astype(np.float32),
        "f": f.astype(np.float32),
    }

    for i in range(high_dim):
        sample[f"x{i + 1}"] = X_high[i].astype(np.float32)

    metadata = {
        "global_id": global_id,
        "F0": float(F0),
        "omega": float(omega),
        "phi": float(phi),

        "x0_burn": float(y_burn0[0]),
        "v0_burn": float(y_burn0[1]),

        "x0_formal": float(y0_formal[0]),
        "v0_formal": float(y0_formal[1]),

        "x_abs_max": x_abs_max,
        "x_range": x_range,
        "v_abs_max": v_abs_max,
        "rms_x": rms_x,
        "large_angle_fraction": large_angle_fraction,
        "nonlinear_gap_rel": nonlinear_gap_rel,
        "corr_x_sinx": corr_x_sinx,

        "difficulty": x_range + 0.25 * v_abs_max + 0.5 * nonlinear_gap_rel,
    }

    return sample, metadata


def accept_sample(metadata):
    required_keys = [
        "difficulty",
        "x_abs_max",
        "x_range",
        "rms_x",
        "large_angle_fraction",
        "nonlinear_gap_rel",
    ]

    for key in required_keys:
        if not np.isfinite(metadata[key]):
            return False

    if metadata["x_abs_max"] > max_abs_x_allowed:
        return False

    if metadata["x_range"] < min_x_range_allowed:
        return False

    if metadata["rms_x"] < min_rms_x_allowed:
        return False

    if metadata["large_angle_fraction"] < min_large_angle_fraction:
        return False

    if metadata["nonlinear_gap_rel"] < min_nonlinear_gap_allowed:
        return False

    return True

samples = []
metadata_list = []

trial = 0
global_id = 1

while len(samples) < num_total:
    trial += 1

    if trial > max_trials:
        raise RuntimeError(
            f"Could not generate enough accepted samples. "
            f"Accepted={len(samples)}, required={num_total}. "
            "You can relax max_abs_x_allowed / min_x_range_allowed / "
            "min_large_angle_fraction / min_nonlinear_gap_allowed."
        )

    try:
        sample, metadata = simulate_one_candidate(global_id=global_id)
    except RuntimeError:
        global_id += 1
        continue

    if accept_sample(metadata):
        samples.append(sample)
        metadata_list.append(metadata)

    global_id += 1

print(f"Accepted {len(samples)} samples after {trial} trials.")

order = np.argsort([m["difficulty"] for m in metadata_list])
blocks = [order[i:i + 10] for i in range(0, len(order), 10)]

split_assignments = {}

for block in blocks:
    block = np.asarray(block, dtype=int)
    rng.shuffle(block)

    test_idx = block[0:1]
    var_idx = block[1:2]
    train_idx = block[2:]

    for idx in train_idx:
        split_assignments[int(idx)] = "train"
    for idx in var_idx:
        split_assignments[int(idx)] = "var"
    for idx in test_idx:
        split_assignments[int(idx)] = "test"

split_counts = {
    "train": sum(1 for v in split_assignments.values() if v == "train"),
    "var": sum(1 for v in split_assignments.values() if v == "var"),
    "test": sum(1 for v in split_assignments.values() if v == "test"),
}

if split_counts["train"] != num_train or split_counts["var"] != num_var or split_counts["test"] != num_test:
    raise RuntimeError(f"Unexpected split counts: {split_counts}")

local_counter = {"train": 1, "var": 1, "test": 1}
metadata_rows = []

for idx, sample in enumerate(samples):
    split = split_assignments[idx]
    local_id = local_counter[split]
    local_counter[split] += 1

    if split == "train":
        save_folder = train_dir
    elif split == "var":
        save_folder = var_dir
    elif split == "test":
        save_folder = test_dir
    else:
        raise ValueError(split)

    filename = f"pendulum_{local_id:03d}.npy"
    save_path = os.path.join(save_folder, filename)
    np.save(save_path, sample)

    row = dict(metadata_list[idx])
    row["split"] = split
    row["local_id"] = local_id
    row["filename"] = filename
    metadata_rows.append(row)


metadata_df = pd.DataFrame(metadata_rows)
metadata_csv = os.path.join(root_dir, "metadata.csv")
metadata_df.to_csv(metadata_csv, index=False)

np.savez(
    os.path.join(root_dir, "lift_mapping_params.npz"),
    high_dim=np.asarray([high_dim], dtype=np.int64),
    max_harmonic=np.asarray([max_harmonic], dtype=np.int64),
    harmonics=harmonics.astype(np.int64),
    phases=phases.astype(np.float32),
    extra_harmonic=np.asarray([extra_harmonic], dtype=np.int64),
    extra_phase=np.asarray([extra_phase], dtype=np.float32),
)

print("Done.")
print(f"Generated {num_train} train files in: {train_dir}")
print(f"Generated {num_var} validation files in: {var_dir}")
print(f"Generated {num_test} test files in: {test_dir}")
print(f"Saved metadata to: {metadata_csv}")
print(f"Saved lift mapping parameters to: {os.path.join(root_dir, 'lift_mapping_params.npz')}")

print("\nSplit counts:")
print(metadata_df["split"].value_counts())

print("\nSplit difficulty summary:")
summary_cols = [
    "difficulty",
    "x_abs_max",
    "x_range",
    "v_abs_max",
    "rms_x",
    "large_angle_fraction",
    "nonlinear_gap_rel",
    "corr_x_sinx",
]

print(
    metadata_df.groupby("split")[summary_cols]
    .agg(["mean", "std", "min", "max"])
)

example_path = os.path.join(train_dir, "pendulum_001.npy")
example = np.load(example_path, allow_pickle=True).item()
df = pd.DataFrame(example)

print("\nExample file:")
print(example_path)
print(df.head())
print(df.shape)
print("Keys:", list(example.keys())[:10], "...", list(example.keys())[-5:])

Accepted 200 samples after 792 trials.
Done.
Generated 160 train files in: pendulum_dataset_periodic_lift_large_angle\train
Generated 20 validation files in: pendulum_dataset_periodic_lift_large_angle\var
Generated 20 test files in: pendulum_dataset_periodic_lift_large_angle\test
Saved metadata to: pendulum_dataset_periodic_lift_large_angle\metadata.csv
Saved lift mapping parameters to: pendulum_dataset_periodic_lift_large_angle\lift_mapping_params.npz

Split counts:
split
train    160
test      20
var       20
Name: count, dtype: int64

Split difficulty summary:
      difficulty                               x_abs_max                      \
            mean       std       min       max      mean       std       min   
split                                                                          
test    4.333499  0.903990  2.687669  6.303532  1.993811  0.436149  1.165173   
train   4.335688  0.871595  2.743203  7.188797  1.990214  0.379908  1.172951   
var     4.329197  0.860914  3.